# Data Mining (AIT3003) - Problem Set 4: Clustering

This notebook contains complete, reproducible code implementations, mathematical derivations, step-by-step algorithms, and visualizations for **Problem Set 4: Clustering (Bài tập 4. Phân cụm dữ liệu)**.

Reference: *Data Mining: Concepts and Techniques (3rd Edition)* by Jiawei Han, Micheline Kamber, and Jian Pei (Morgan Kaufmann, Chapter 10).


## Problem P1: Cell-Based Entropy for Grid-Based Data

> **Prompt:** Cho một tập dữ liệu gồm 8 điểm dữ liệu phân bố trong lưới $4 \times 4$ như hình vẽ. Hãy tính entropy $H$ của phân bố các điểm dữ liệu đó theo phương pháp cell-based entropy.

### Mathematical Formulation
- Grid dimensions: $4 \times 4 = 16$ cells.
- Total points: $N = 8$.
- Probability of cell $c_i$: $p_i = \frac{n_i}{N}$.
- Non-empty cells:
  - $\text{Cell}(1, 4)$: $n_1 = 2 \implies p_1 = \frac{2}{8} = \frac{1}{4} = 0.25$
  - $\text{Cell}(3, 1)$: $n_2 = 4 \implies p_2 = \frac{4}{8} = \frac{1}{2} = 0.50$
  - $\text{Cell}(4, 3)$: $n_3 = 2 \implies p_3 = \frac{2}{8} = \frac{1}{4} = 0.25$
- Shannon Entropy (base 2):
  $$H = -\sum_{i=1}^{m} p_i \log_2(p_i) = -\left[ 0.25(-2) + 0.50(-1) + 0.25(-2) \right] = -[-1.5] = 1.5000 \text{ bits}$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

grid_counts = np.zeros((4, 4), dtype=int)
grid_counts[0, 3] = 2  # Cell (row 1, col 4)
grid_counts[2, 0] = 4  # Cell (row 3, col 1)
grid_counts[3, 2] = 2  # Cell (row 4, col 3)

total_points = 8
p_vals = [2/8, 4/8, 2/8]
h_base2 = -sum(p * np.log2(p) for p in p_vals)
h_nat = -sum(p * np.log(p) for p in p_vals)
h_norm = h_base2 / np.log2(16)

print(f"Point counts in 4x4 grid:\n{grid_counts}")
print(f"Cell probabilities: {p_vals}")
print(f"Entropy H (base 2)     : {h_base2:.4f} bits")
print(f"Entropy H (base e)     : {h_nat:.4f} nats")
print(f"Normalized Entropy     : {h_norm:.4f} ({h_norm*100:.1f}% of max)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

im = axes[0].imshow(grid_counts, cmap="YlOrRd", vmin=0, vmax=5)
plt.colorbar(im, ax=axes[0], label="Point Count")
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].set_xticklabels([f"Col {j+1}" for j in range(4)])
axes[0].set_yticklabels([f"Row {i+1}" for i in range(4)])

for i in range(4):
    for j in range(4):
        c = grid_counts[i, j]
        axes[0].text(j, i, f"{c} pts\n(p={c/8:.2f})" if c > 0 else "0",
                     ha="center", va="center", color="white" if c >= 4 else "black", fontweight="bold")

axes[0].set_title("4x4 Grid Density Distribution")

terms = [-(p * np.log2(p)) for p in p_vals]
labels = ["Cell (1,4)", "Cell (3,1)", "Cell (4,3)"]
axes[1].bar(labels, terms, color=["#1f77b4", "#ff7f0e", "#2ca02c"], edgecolor="black", width=0.5)
for idx, val in enumerate(terms):
    axes[1].text(idx, val/2, f"+{val:.2f} bits", ha="center", va="center", color="white", fontweight="bold")

axes[1].set_ylabel("Entropy Contribution -p_i log2(p_i)")
axes[1].set_title(f"Cell Entropy Breakdown (Total H = {h_base2:.1f} bits)")
axes[1].grid(True, axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


## Problem P2: K-Means Clustering on 2D Spatial Points

> **Prompt:** Giả sử rằng nhiệm vụ khai thác dữ liệu là nhóm điểm (với $(x, y)$ biểu diễn vị trí) thành ba cụm, trong đó các điểm là:
> $$A_1(2,10), A_2(2,5), A_3(8,4), B_1(5,8), B_2(7,5), B_3(6,4), C_1(1,2), C_2(4,9)$$
> Hàm khoảng cách là khoảng cách Euclidean. Giả sử ban đầu chúng ta gán $A_1, B_1$ và $C_1$ làm tâm của mỗi cụm, tương ứng. Sử dụng thuật toán k-means để chỉ ra:
> - a) Ba tâm cụm sau vòng thực thi đầu tiên.
> - b) Ba cụm cuối cùng.


In [ ]:
points = {
    'A1': np.array([2.0, 10.0]),
    'A2': np.array([2.0, 5.0]),
    'A3': np.array([8.0, 4.0]),
    'B1': np.array([5.0, 8.0]),
    'B2': np.array([7.0, 5.0]),
    'B3': np.array([6.0, 4.0]),
    'C1': np.array([1.0, 2.0]),
    'C2': np.array([4.0, 9.0]),
}

centroids = [points['A1'].copy(), points['B1'].copy(), points['C1'].copy()]
iteration = 0
prev_clusters = None
history = []

while True:
    iteration += 1
    clusters = {1: [], 2: [], 3: []}
    for name, pt in points.items():
        dists = [np.linalg.norm(pt - c) for c in centroids]
        c_id = int(np.argmin(dists) + 1)
        clusters[c_id].append(name)
    
    history.append({'iteration': iteration, 'centroids': [c.copy() for c in centroids], 'clusters': clusters})
    print(f"Iteration {iteration} Clusters: {clusters}")
    
    if clusters == prev_clusters:
        print(f"Converged at iteration {iteration}!")
        break
    
    prev_clusters = {k: list(v) for k, v in clusters.items()}
    centroids = [np.mean([points[p] for p in clusters[k]], axis=0) for k in [1, 2, 3]]
    for k, c in enumerate(centroids):
        print(f"  New centroid m{k+1} = ({c[0]:.4f}, {c[1]:.4f})")


In [ ]:
# Print answers to part a and b
r1_clusters = history[0]['clusters']
m1_r1 = np.mean([points[p] for p in r1_clusters[1]], axis=0)
m2_r1 = np.mean([points[p] for p in r1_clusters[2]], axis=0)
m3_r1 = np.mean([points[p] for p in r1_clusters[3]], axis=0)

print("=== ANSWERS FOR PROBLEM P2 ===")
print("a) Ba tâm cụm sau vòng thực thi đầu tiên:")
print(f"   m1^(1) = ({m1_r1[0]:.2f}, {m1_r1[1]:.2f})")
print(f"   m2^(1) = ({m2_r1[0]:.2f}, {m2_r1[1]:.2f})")
print(f"   m3^(1) = ({m3_r1[0]:.2f}, {m3_r1[1]:.2f})")

final_clusters = history[-1]['clusters']
print("\nb) Ba cụm cuối cùng:")
for k in [1, 2, 3]:
    c_pts = final_clusters[k]
    c_mean = np.mean([points[p] for p in c_pts], axis=0)
    print(f"   Cụm {k}: {c_pts} -> Tâm cụm m{k}* = ({c_mean[0]:.4f}, {c_mean[1]:.4f})")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=120)
colors = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c"}

for idx, it in enumerate(history[:3]):
    ax = axes[idx]
    for c_id in [1, 2, 3]:
        pts = np.array([points[p] for p in it['clusters'][c_id]])
        if len(pts) > 0:
            ax.scatter(pts[:, 0], pts[:, 1], color=colors[c_id], s=100, edgecolors="black", label=f"Cluster {c_id}")
            for p_name in it['clusters'][c_id]:
                ax.annotate(p_name, (points[p_name][0]+0.15, points[p_name][1]+0.15), fontweight="bold")
    for c_id, cent in enumerate(it['centroids']):
        ax.scatter(cent[0], cent[1], color=colors[c_id+1], marker="X", s=200, edgecolors="black", label=f"Centroid m{c_id+1}")
    ax.set_title(f"Iteration {it['iteration']}")
    ax.set_xlim(0, 9.5)
    ax.set_ylim(1, 11)
    ax.grid(True, linestyle=":", alpha=0.6)
    if idx == 0:
        ax.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()


## Problem P3: Agglomerative Hierarchical Clustering (Single & Complete Link)

> **Prompt:** Bảng dưới đây là ma trận khoảng cách của 6 đối tượng:
> - a) Chỉ ra kết quả cuối cùng của phân cụm phân cấp với liên kết đơn (single link) bằng cách vẽ sơ đồ phân cấp (dendrogram).
> - b) Chỉ ra kết quả cuối cùng của phân cụm phân cấp với liên kết hoàn chỉnh (complete link) bằng cách vẽ sơ đồ phân cấp.


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

labels = ['A', 'B', 'C', 'D', 'E', 'F']
n = 6
dist_mat = np.zeros((n, n))
lower_entries = {
    (1, 0): 0.12,
    (2, 0): 0.51, (2, 1): 0.25,
    (3, 0): 0.84, (3, 1): 0.16, (3, 2): 0.14,
    (4, 0): 0.28, (4, 1): 0.77, (4, 2): 0.70, (4, 3): 0.45,
    (5, 0): 0.34, (5, 1): 0.61, (5, 2): 0.93, (5, 3): 0.20, (5, 4): 0.67
}
for (i, j), v in lower_entries.items():
    dist_mat[i, j] = v
    dist_mat[j, i] = v

condensed = squareform(dist_mat)
Z_single = linkage(condensed, method='single')
Z_complete = linkage(condensed, method='complete')

print("Single Link Merges:")
for idx, row in enumerate(Z_single):
    print(f"  Step {idx+1}: Merge {int(row[0])} and {int(row[1])} at dist={row[2]:.4f}")

print("\nComplete Link Merges:")
for idx, row in enumerate(Z_complete):
    print(f"  Step {idx+1}: Merge {int(row[0])} and {int(row[1])} at dist={row[2]:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), dpi=120)

axes[0].set_title("(a) Single Link Dendrogram")
dendro_s = dendrogram(Z_single, labels=labels, ax=axes[0], color_threshold=0, above_threshold_color='#1f77b4')
axes[0].set_ylabel("Distance")
axes[0].grid(True, axis="y", linestyle=":", alpha=0.6)
for i, d, c in zip(dendro_s["icoord"], dendro_s["dcoord"], dendro_s["color_list"]):
    x = 0.5 * sum(i[1:3])
    y = d[1]
    if y > 0:
        axes[0].text(x, y + 0.01, f"{y:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#1f77b4")

axes[1].set_title("(b) Complete Link Dendrogram")
dendro_c = dendrogram(Z_complete, labels=labels, ax=axes[1], color_threshold=0, above_threshold_color='#d62728')
axes[1].set_ylabel("Distance")
axes[1].grid(True, axis="y", linestyle=":", alpha=0.6)
for i, d, c in zip(dendro_c["icoord"], dendro_c["dcoord"], dendro_c["color_list"]):
    x = 0.5 * sum(i[1:3])
    y = d[1]
    if y > 0:
        axes[1].text(x, y + 0.02, f"{y:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#d62728")

plt.tight_layout()
plt.show()
